# Explore the Kafka mirror

This notebook reads data from the mirror broker: which topics it holds, the latest messages in a topic, what arrived recently, a live view of new records, and the consumer groups on the mirror. The work is done by the `neon_sage_edge` package in this repository; the notebook only calls it, so the same functions can also be used from scripts.

If your topics' values are written with a schema registry (Avro or JSON Schema), the package can deserialize them into readable records. Point it at the registry in the connection settings; without one, values are shown as text or binaries.

It only **reads**.

The notebook connects to the mirror's external listener, `localhost:9094` by default. If Jupyter runs on another machine, set `MIRROR_BOOTSTRAP` to `<mirror-host>:9094` and make sure the broker advertises that host.

In [1]:
# Installs the package (and its Avro / JSON Schema support) from this repository.
# If pip reports "externally-managed-environment", start Jupyter from a virtual environment instead.
%pip install --quiet -e "..[registry]"

Note: you may need to restart the kernel to use updated packages.


## Connection settings

Defaults match the mirror setup. Override any of them with environment variables: `MIRROR_BOOTSTRAP`, `MIRROR_USERNAME`, `MIRROR_PASSWORD`, and `MIRROR_SECURITY_PROTOCOL`.

**Schema registry and deserializers (optional).** Every message's key and value go through a deserializer, chosen with `VALUE_FORMAT` and `KEY_FORMAT`:

- `avro` or `json`: the schema registry's Avro or JSON Schema deserializer. It fetches the schema each record was written with, so older and newer schema versions both work.
- `string`: plain UTF-8 text, with no registry involved.

Set `SCHEMA_REGISTRY_URL` (no login is used) and values default to `avro`; keys default to `string`. If this machine can't resolve the registry's name, also set `SCHEMA_REGISTRY_IP` to the address to use, as `curl --resolve` does; that mapping applies inside this notebook only.

Instead of environment variables, you can also build the settings in the cell below, as the commented example shows.

In [ ]:
import os
from getpass import getpass

import pandas as pd
from neon_sage_edge import (KafkaSettings, MessageDecoder, MirrorReader, RegistrySettings, expand_json, format_row,
                            make_registry_client, per_minute_counts, registry_subjects, show_notes, time_range)

show_notes()  # print the package's notes (e.g. records it couldn't deserialize) as plain lines
pd.set_option("display.max_colwidth", 120)

kafka = KafkaSettings.from_env(prompt=getpass)
registry = RegistrySettings(url="",
                             ip="",
                             value_format="avro",
                            )

registry_client = make_registry_client(registry)
reader = MirrorReader.connect(kafka, MessageDecoder.from_settings(registry, registry_client))

print("Mirror:", kafka.describe())
print("Schema registry:", registry.describe())

## 1. Topics on the mirror

Your mirrored topics, with partition counts and how many messages each holds.

In [3]:
reader.topics()

,topic,partitions,messages
0,reading.sensor.csat3,1,0
1,reading.sensor.csat3b,1,6757443
2,reading.sensor.li7200_raw,1,6753850


## 2. Pick a topic

Set `TOPIC` to one of the topics above. The table shows, per partition, the oldest retained offset (`low`), the next offset to be written (`high`), and the difference.

In [4]:
TOPIC = os.environ.get("NOTEBOOK_TOPIC", "reading.sensor.csat3b")
reader.watermarks(TOPIC)

,partition,low,high,messages
0,0,0,6757443,6757443


## 3. Latest messages

In [5]:
latest = reader.peek(TOPIC, n=20)
latest

,partition,offset,timestamp,key,value,headers
0,0,6757423,2026-10-09 16:35:23.939000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-09 16:35:12.500000+00:00, 'ux_wind_speed': -0.3131...",None
1,0,6757424,2026-10-09 16:35:23.939000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-09 16:35:12.550000+00:00, 'ux_wind_speed': -0.3748...",None
2,0,6757425,2026-10-09 16:35:23.939000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-09 16:35:12.600000+00:00, 'ux_wind_speed': -0.2431...",None
3,0,6757426,2026-10-09 16:35:23.939000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-09 16:35:12.650000+00:00, 'ux_wind_speed': -0.0819...",None
4,0,6757427,2026-10-09 16:35:23.939000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-09 16:35:12.700000+00:00, 'ux_wind_speed': -0.1156...",None
5,0,6757428,2026-10-09 16:35:23.939000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-09 16:35:12.750000+00:00, 'ux_wind_speed': -0.2300...",None
6,0,6757429,2026-10-09 16:35:23.939000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-09 16:35:12.800000+00:00, 'ux_wind_speed': -0.1584...",None
7,0,6757430,2026-10-09 16:35:23.939000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-09 16:35:12.850000+00:00, 'ux_wind_speed': -0.3569...",None
8,0,6757431,2026-10-09 16:35:23.939000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-09 16:35:12.900000+00:00, 'ux_wind_speed': -0.2861...",None
9,0,6757432,2026-10-09 16:35:23.939000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-09 16:35:12.950000+00:00, 'ux_wind_speed': -0.3000...",None


If the values are JSON objects, or records from the schema registry, this spreads their fields into columns:

In [6]:
expand_json(latest)

,partition,offset,timestamp,key,headers,source_id,site_id,readout_time,ux_wind_speed,uy_wind_speed,uz_wind_speed,sonic_temperature,diagnostic,pitch,roll,board_temp,board_humidity
0,0,6757423,2026-10-09 16:35:23.939000+00:00,59483,None,59483,CPER,2026-10-09 16:35:12.500000+00:00,-0.313157,5.003365,-0.088685,24.162571,0,1.987496,1.500393,31.8125,28.999611
1,0,6757424,2026-10-09 16:35:23.939000+00:00,59483,None,59483,CPER,2026-10-09 16:35:12.550000+00:00,-0.374891,4.936113,-0.037450,24.146549,0,1.987496,1.500393,31.8125,28.999611
2,0,6757425,2026-10-09 16:35:23.939000+00:00,59483,None,59483,CPER,2026-10-09 16:35:12.600000+00:00,-0.243190,4.929592,-0.050213,24.218929,0,1.987496,1.500393,31.8125,28.999611
3,0,6757426,2026-10-09 16:35:23.939000+00:00,59483,None,59483,CPER,2026-10-09 16:35:12.650000+00:00,-0.081961,4.852390,-0.155252,24.215151,0,1.987496,1.500393,31.8125,28.999611
4,0,6757427,2026-10-09 16:35:23.939000+00:00,59483,None,59483,CPER,2026-10-09 16:35:12.700000+00:00,-0.115624,4.844035,-0.170000,24.103420,0,1.987496,1.500393,31.8125,28.999611
5,0,6757428,2026-10-09 16:35:23.939000+00:00,59483,None,59483,CPER,2026-10-09 16:35:12.750000+00:00,-0.230015,4.767322,-0.119438,24.117611,0,1.987496,1.500393,31.8125,28.999611
6,0,6757429,2026-10-09 16:35:23.939000+00:00,59483,None,59483,CPER,2026-10-09 16:35:12.800000+00:00,-0.158455,4.818099,-0.158216,24.096649,0,1.987496,1.500393,31.8125,28.999611
7,0,6757430,2026-10-09 16:35:23.939000+00:00,59483,None,59483,CPER,2026-10-09 16:35:12.850000+00:00,-0.356971,4.706380,-0.127114,24.295990,0,1.987496,1.500393,31.8125,28.999611
8,0,6757431,2026-10-09 16:35:23.939000+00:00,59483,None,59483,CPER,2026-10-09 16:35:12.900000+00:00,-0.286153,4.331586,-0.053856,24.287201,0,1.987496,1.500393,31.8125,28.999611
9,0,6757432,2026-10-09 16:35:23.939000+00:00,59483,None,59483,CPER,2026-10-09 16:35:12.950000+00:00,-0.300098,4.365672,-0.056941,24.423130,0,1.987496,1.500393,31.8125,28.999611


## 4. What arrived recently

Messages from the last 15 minutes by record timestamp, and how many arrived each minute. MirrorMaker keeps the original timestamps, so these reflect when records were produced on the source.

In [7]:
recent = reader.read_since(TOPIC, minutes=15, max_messages=5_000)
print(f"{len(recent)} messages in the last 15 minutes")
per_minute_counts(recent)

Stopped at 5000 messages (the max_messages limit); pass a larger max_messages to read the rest.


5000 messages in the last 15 minutes


,messages
timestamp,
2026-10-09 16:20:00+00:00,800
2026-10-09 16:21:00+00:00,2050
2026-10-09 16:22:00+00:00,1220
2026-10-09 16:23:00+00:00,930


The earliest and latest `readout_time` in those messages, next to their Kafka timestamps. `readout_time` comes from inside each record (when the reading was taken), while the Kafka timestamp is when the record was produced, so comparing the two shows how long readings take to reach Kafka. Pass another field name to `time_range` to do the same for it.

In [8]:
time_range(recent, "readout_time")

,messages,min,max,span
field,,,,
readout_time,5000,2026-10-09 16:20:17+00:00,2026-10-09 16:23:25.950000+00:00,0 days 00:03:08.950000
timestamp (Kafka),5000,2026-10-09 16:20:47.691000+00:00,2026-10-09 16:23:50.999000+00:00,0 days 00:03:03.308000


## 5. Watch new messages arrive

Prints each new message for 30 seconds, and keeps them in `live`. Produce something on the source while this runs to see replication happen.

In [9]:
live = reader.watch(TOPIC, seconds=10, on_message=lambda row: print(format_row(row)))

partition 0  offset 6757443  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 6757444  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 6757445  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 6757446  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 6757447  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 6757448  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 6757449  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 6757450  key='59483'  value="{'source_id': '59483', 'site_id': 

## 6. Schema registry

Only does something when a registry is configured. Lists the registry's subjects with their latest schema, which helps pick `VALUE_FORMAT`: an `AVRO` subject needs `avro`, a `JSON` one `json`. Protobuf isn't supported, because its deserializer needs the generated message classes.

In [10]:
registry_subjects(registry_client)

,subject,version,schema_id,type
0,aepg600m,1,108,AVRO
1,aepg600m_heated,1,194,AVRO
2,aquatroll200,1,134,AVRO
3,ce318n_metrics,1,120,AVRO
4,ce318n_raw,1,99,AVRO
...,...,...,...,...
95,tdr315n_metrics,2,208,AVRO
96,tdr315n_raw,2,207,AVRO
97,windmonitorhd,1,47,AVRO
98,windobserverii,1,107,AVRO
